# Day 3 · 5교시 [실습 보조] Claude Agent SDK — `05_agent_sdk`

## 이 노트북이 답하는 것

**Claude Agent SDK = 3일 동안 쓴 Claude Code 를, 터미널 대신 파이썬 코드에서 부르는 것.**
에이전트를 *만드는* 게 아니라, 이미 완성된 것을 **물려받아** 내 것만 위에 얹는다.

| 순서 | 내용 | 교안 |
|------|------|------|
| 1 | 환경 점검 — 무엇이 준비돼 있나 | 5.5 |
| 2 | 에이전트를 «함수»로 감싼다 (`claude -p` 래퍼) | 5.4 |
| 3 | 🔥 Windows 함정 — argv vs stdin | 5.4 |
| 4 | `query()` — 같은 함수의 SDK 판 | 5.5 |
| 5 | `ClaudeAgentOptions` — 파일로 하던 걸 인자로 | 5.3 |
| 6 | 옵션 한 줄 더 — MCP·서브에이전트·커스텀 도구 | 5.6 |
| 7 | 관통 프로젝트 — 요약 단계를 SDK로 | 5.5 |

> **준비물은 한 줄이다**: `pip install claude-agent-sdk` (Python 3.10+).
> **별도 API 키는 필요 없다** — SDK 는 내부적으로 `claude` CLI 를 띄우고 **Claude Code 가 이미 쓰는
> 로그인 자격증명을 그대로 쓴다**(공식 문서: 인증 우선순위 ⑥ `/login` 구독 OAuth).
> 환경에 `ANTHROPIC_API_KEY` 가 남아 있으면 **그쪽이 우선**해 종량 과금이 나가니 주의.
>
> ⚠️ SDK 는 변동이 잦다(패키지명이 *Claude Code SDK → Claude Agent SDK* 로 바뀐 이력).
> 옵션·필드는 공식 문서로 최신 확인: <https://code.claude.com/docs/ko/agent-sdk/overview>


## 1. 환경 점검 — 무엇이 준비돼 있나

SDK 가 없어도 이 노트북은 **끝까지 무에러로 열린다**(그 칸은 참고 코드만 출력).

In [ ]:
import os, shutil, inspect, subprocess

CLAUDE = shutil.which("claude")          # Windows 에서는 claude.CMD 를 찾아 준다
HAVE_CLI = CLAUDE is not None
try:
    from claude_agent_sdk import query, ClaudeAgentOptions
    HAVE_SDK = True
except Exception:
    HAVE_SDK = False

print("claude CLI          :", CLAUDE or "없음 (Day 1 설치분)")
print("claude-agent-sdk    :", HAVE_SDK, "" if HAVE_SDK else "→ pip install claude-agent-sdk")
print("ANTHROPIC_API_KEY   :", bool(os.getenv("ANTHROPIC_API_KEY")),
      "← True 면 구독이 아니라 «종량 과금»으로 나간다" if os.getenv("ANTHROPIC_API_KEY") else "← 없어도 된다(구독 로그인 사용)")

## 2. 에이전트를 «함수»로 감싼다 — `claude -p` 래퍼 (교안 5.4)

SDK 를 깔기 전에 **SDK 가 하는 일의 뼈대**를 10줄로 만들어 본다.
4교시의 헤드리스 `claude -p` 를 파이썬 함수 안에 넣는 것 — 설치도 키도 필요 없다.

In [ ]:
def ask_agent(prompt: str) -> str:
    """에이전트를 «함수»로 감싼다 — 프롬프트는 stdin 으로 넘긴다(아래 3절 참고)."""
    r = subprocess.run([CLAUDE, "-p"], input=prompt,
                       capture_output=True, text=True, encoding="utf-8")
    return r.stdout.strip()

def summarize_cli(items: list[str]) -> str:
    return ask_agent("다음 수집 항목을 한국어 3문장으로 요약하라. 사실만 쓴다.\n"
                     + "\n".join("- " + i for i in items))

ITEMS = ["MCP 표준이 확산 중", "에이전트 병렬 워크플로가 늘어남", "프롬프트 인젝션 관심 증가"]

if HAVE_CLI:
    print(summarize_cli(ITEMS))
else:
    print("[참고 코드] claude CLI 가 없다 — Day 1 설치분 확인\n")
    print(inspect.getsource(summarize_cli))

## 3. 🔥 Windows 함정 — 여러 줄 프롬프트는 stdin 으로

`claude` 는 Windows 에서 **`claude.CMD` 셸 래퍼**다. 줄바꿈이 든 문자열을
`subprocess.run([CLAUDE, "-p", prompt])` 처럼 **인자로** 넘기면 **첫 줄에서 잘린다.**
아래에서 두 방식을 나란히 돌려 직접 확인한다.

In [ ]:
PROBE = ("아래 지시를 따르라.\n"
         "두 번째 줄에 있는 단어 BANANA 만 그대로 출력하라. 다른 말 금지.")

if HAVE_CLI:
    argv_out = subprocess.run([CLAUDE, "-p", PROBE], capture_output=True,
                              text=True, encoding="utf-8").stdout.strip()
    stdin_out = subprocess.run([CLAUDE, "-p"], input=PROBE, capture_output=True,
                               text=True, encoding="utf-8").stdout.strip()
    print("--- argv  방식:", argv_out[:120])
    print("--- stdin 방식:", stdin_out[:120])
    print("\n→ argv 쪽은 2줄째 지시가 통째로 사라진다. 여러 줄은 반드시 input=(stdin).")
else:
    print("[참고] argv 방식은 첫 줄에서 잘린다 → input=prompt (stdin) 으로 넘길 것")

## 4. `query()` — 같은 함수의 SDK 판 (교안 5.5)

`query(prompt, options)` 는 한 번 실행하고 **메시지를 비동기로 흘려주는** 함수다.
마지막에 오는 `ResultMessage` 의 `.result` 가 최종 답이다.

> 🔥 **주피터 안에서는 이 셀을 직접 못 돌린다(Windows).** SDK 는 내부에서 `claude` CLI 를
> **서브프로세스로 띄우는데**, 주피터 커널의 이벤트 루프에서는 그 spawn 이 실패한다
> (`CLIConnectionError: Failed to start Claude Code`). **실측으로 확인한 제약**이다.
> 그래서 아래 셀은 코드를 `05_sdk.py` 로 **파일에 쓰고 별도 프로세스로 실행**한다 — 교안 5.5 와 같은 방식.


In [ ]:
import sys, pathlib

SDK_SRC = r'''# day3/05_sdk.py  (Python 3.10+) — 교안 5.5
# 관통 프로젝트 Summarizer — SDK 임베드 버전
import asyncio, os
from claude_agent_sdk import query, ClaudeAgentOptions

print("ANTHROPIC_API_KEY 설정 여부:", bool(os.getenv("ANTHROPIC_API_KEY")))

async def summarize_sdk(items):
    opts = ClaudeAgentOptions(
        system_prompt="너는 우리 파이프라인의 요약 담당이다. 사실만, 3문장.",
        allowed_tools=["Read", "Grep"],     # 최소 권한(Day 1 5교시 원칙을 코드로)
        permission_mode="default",
    )
    async for message in query(prompt="\n".join("- " + i for i in items), options=opts):
        if hasattr(message, "result"):      # 마지막 ResultMessage 가 최종 답
            return message.result
    return ""

if __name__ == "__main__":
    print(asyncio.run(summarize_sdk(
        ["MCP 표준이 확산 중", "에이전트 병렬 워크플로가 늘어남", "프롬프트 인젝션 관심 증가"])))
'''

pathlib.Path("05_sdk.py").write_text(SDK_SRC, encoding="utf-8")
print("05_sdk.py 를 썼다 —", len(SDK_SRC), "bytes")

if HAVE_SDK:
    r = subprocess.run([sys.executable, "05_sdk.py"], capture_output=True,
                       text=True, encoding="utf-8")
    print(r.stdout.strip() or r.stderr.strip()[:800])
else:
    print("[참고] pip install claude-agent-sdk 후 터미널에서: python day3/05_sdk.py")

## 5. `ClaudeAgentOptions` — 파일로 하던 걸 인자로 (교안 5.3)

지난 3일 동안 **파일**로 정하던 것이 SDK 에선 그대로 **인자**가 된다. 새 개념이 0개인 이유다.

| 지난 3일 (파일·명령) | SDK (코드 인자) |
|---|---|
| `CLAUDE.md` | `system_prompt=` |
| `settings.json` 의 `allow` | `allowed_tools=` |
| 권한 모드 · Plan 모드 | `permission_mode=` |
| `.claude/agents/*.md` | `agents=` |
| `claude mcp add ...` | `mcp_servers=` |
| 훅 | `hooks=` |
| `claude --resume` | `resume=session_id` |

In [ ]:
def build_options():
    """harness(설정·권한·규칙)를 «파일»이 아니라 «코드»로 조립한다."""
    return ClaudeAgentOptions(
        system_prompt="너는 한국어 요약기다. 사실만, 두 문장 이내.",
        allowed_tools=["Read", "Grep"],
        permission_mode="default",          # default / plan / acceptEdits / bypassPermissions
        model="sonnet",                     # 모델 티어링(Day 2 1교시)도 코드로
    )

if HAVE_SDK:
    print("옵션 객체 생성 OK:", type(build_options()).__name__)
else:
    print(inspect.getsource(build_options))

## 6. 옵션 한 줄 더 — MCP · 서브에이전트 · 커스텀 도구 (교안 5.6)

Day 2 에서 **CLI 로** 붙였던 것들은 SDK 에선 **옵션 한 줄**이다. 자리만 옮긴 것.
CLI 에 없던 것 하나: `@tool` 로 만든 **인-프로세스 도구**(외부 프로세스 없이 내 함수가 도구가 된다).

*(아래는 형태만 — 시그니처는 공식 문서로 확인)*

In [ ]:
SNIPPET = """
from claude_agent_sdk import tool, create_sdk_mcp_server, ClaudeAgentOptions

@tool("word_count", "텍스트 단어 수", {"text": str})
async def word_count(args):
    n = len(args["text"].split())
    return {"content": [{"type": "text", "text": f"{n} words"}]}

server = create_sdk_mcp_server(name="tools", tools=[word_count])

options = ClaudeAgentOptions(
    mcp_servers={"tools": server,
                 "playwright": {"command": "npx", "args": ["@playwright/mcp@latest"]}},
    agents={"reviewer": {"description": "코드 리뷰 전용",
                         "prompt": "변경을 심각도순으로 리뷰",
                         "tools": ["Read", "Grep"]}},
    allowed_tools=["Agent", "Read", "Grep", "mcp__tools__word_count"],
)
"""
print("[참고 코드] @tool → create_sdk_mcp_server → ClaudeAgentOptions")
print(SNIPPET)

## 7. 관통 프로젝트 — 요약 단계를 SDK로 (계약은 그대로)

Day 2 에서 정한 인터페이스 `summarize(items) -> str` 를 **그대로 지키고 구현만 갈아 끼운다.**
그러면 파이프라인(수집→저장→요약→알림)의 나머지는 손대지 않아도 된다.

In [ ]:
import sys

def summarize(items: list[str]) -> str:
    """파이프라인이 부르는 «동기» 계약 — 안에서만 구현을 고른다.

    SDK 판은 별도 프로세스로 부른다(4절의 주피터 제약). 스크립트로 돌릴 때는
    05_sdk.py 의 summarize_sdk 를 import 해서 asyncio.run 으로 바로 부르면 된다.
    """
    if HAVE_SDK:
        r = subprocess.run([sys.executable, "05_sdk.py"], capture_output=True,
                           text=True, encoding="utf-8")
        if r.returncode == 0 and r.stdout.strip():
            return r.stdout.strip().splitlines()[-1]
    if HAVE_CLI:
        return summarize_cli(items)           # SDK 가 없으면 CLI 래퍼로 폴백
    return ""

print("계약 유지:", inspect.signature(summarize))
print(summarize(ITEMS))

## 실습 정리

- **SDK = Claude Code 를 물려받는 것.** 도구(Read·Bash·WebSearch…)도, 세션도, **로그인도** 이미 있는 것을 그대로 쓴다 — 그래서 준비물이 `pip install` 한 줄이다.
- **새 개념은 0개.** `CLAUDE.md`·`settings.json`·`.claude/agents/`·`mcp add`·훅이 전부 `ClaudeAgentOptions` 의 인자로 내려왔다(harness 를 파일에서 코드로).
- **CLI 래퍼(2절) → SDK(4절)** 로 넘어가며 얻는 것: 메시지 객체 스트림 · `allowed_tools` · `permission_mode` · `resume` · `hooks` · `agents`.
- 🔥 Windows: 여러 줄 프롬프트는 **stdin** 으로. argv 로 넘기면 첫 줄에서 잘린다.
- 💸 환경에 `ANTHROPIC_API_KEY` 가 남아 있으면 구독이 아니라 **종량 과금**으로 나간다.
- ⚠️ 변동이 잦은 영역 → 실습 전 공식 문서로 최신화.